<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS04_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS04 — Clustering dengan K-Means pada Data Properti

Jobsheet asli menerapkan K-Means pada Iris, data blob, digit, dan kuantisasi warna.
Versi PBL ini mengelompokkan **properti** berdasarkan luas tanah, luas bangunan, jumlah
kamar, dan jumlah kamar mandi, lalu memeriksa apakah klaster yang terbentuk sejalan
dengan kategori harga Murah/Sedang/Mahal.

## Langkah 1 — Muat Data dan Siapkan Fitur

Fitur yang diklaster adalah empat atribut fisik properti. `harga` **tidak** dipakai
sebagai fitur (hanya untuk mengevaluasi hasil klaster), karena jika harga ikut masuk,
klaster akan trivial: semua rumah mahal berkumpul sendiri.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score,
                             normalized_mutual_info_score)

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42

df = pd.read_csv('dataset/properti_multi_kota.csv').drop_duplicates()
df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
df = df[df['luas_bangunan'].isna() |
        ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
df = df[df['harga'].between(50_000_000, 100_000_000_000)]
df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]

FITUR = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
df[FITUR] = df[FITUR].fillna(df[FITUR].median())

q1, q2 = df['harga'].quantile([1 / 3, 2 / 3])
df['kelas_harga'] = np.where(df['harga'] < q1, 'Murah',
                     np.where(df['harga'] <= q2, 'Sedang', 'Mahal'))

X = StandardScaler().fit_transform(df[FITUR])
print(f'Data siap klaster: {X.shape}')
print('Fitur:', FITUR)

Data siap klaster: (56170, 4)
Fitur: ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']


## Langkah 2 — Menentukan Jumlah Klaster (Elbow dan Silhouette)

Dua indikator dipakai bersamaan:

- **Inertia (SSE)** — menurun terus saat `k` naik, jadi titik "siku" yang dicari.
- **Silhouette** — mengukur seberapa rapat anggota klaster dan seberapa jauh dari klaster
  lain. Nilai mendekati 1 lebih baik.

Perhitungan memakai sampel 10.000 baris agar cepat; pola kurvanya sama dengan data penuh.

In [2]:
SAMPEL = df.sample(10_000, random_state=RANDOM_STATE)
X_sampel = StandardScaler().fit_transform(SAMPEL[FITUR])

ks = range(2, 11)
inertia, silhouette = [], []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_sampel)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_sampel, km.labels_, sample_size=3000,
                                       random_state=RANDOM_STATE))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(list(ks), inertia, 'o-', color='steelblue')
ax[0].set_xlabel('Jumlah klaster (k)')
ax[0].set_ylabel('Inertia (SSE)')
ax[0].set_title('Elbow method')

ax[1].plot(list(ks), silhouette, 'o-', color='darkorange')
ax[1].set_xlabel('Jumlah klaster (k)')
ax[1].set_ylabel('Silhouette score')
ax[1].set_title('Silhouette per k')

plt.tight_layout()
plt.show()

pd.DataFrame({'k': list(ks), 'inertia': inertia, 'silhouette': silhouette}).round(4)

,k,inertia,silhouette
0,2,23831.3752,0.6431
1,3,17833.3288,0.5138
2,4,14988.0076,0.5224
3,5,12894.4213,0.4371
4,6,11457.3272,0.4423
5,7,10362.5668,0.4213
6,8,9571.8073,0.4470
7,9,8823.8492,0.4551
8,10,8303.3306,0.4448


## Langkah 3 — Latih K-Means pada Seluruh Data

`k = 3` dipilih agar hasilnya bisa dibandingkan langsung dengan tiga kelas harga
(Murah/Sedang/Mahal). Model dilatih pada 56.170 baris dengan `n_init=10` supaya hasil
tidak bergantung pada inisialisasi awal.

In [3]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X)
df['klaster'] = kmeans.labels_

sil = silhouette_score(X, df['klaster'], sample_size=5000, random_state=RANDOM_STATE)
dbi = davies_bouldin_score(X, df['klaster'])
chi = calinski_harabasz_score(X, df['klaster'])

print(f'Inertia             : {kmeans.inertia_:,.0f}')
print(f'Silhouette          : {sil:.4f}')
print(f'Davies-Bouldin      : {dbi:.4f}  (semakin kecil semakin baik)')
print(f'Calinski-Harabasz   : {chi:,.0f}')
print(f'Ukuran klaster      : {np.bincount(df["klaster"]).tolist()}')

Inertia             : 100,165
Silhouette          : 0.5247
Davies-Bouldin      : 1.0484  (semakin kecil semakin baik)
Calinski-Harabasz   : 34,911
Ukuran klaster      : [12878, 40761, 2531]


## Langkah 4 — Profil Setiap Klaster

Rata-rata atribut dan median harga menunjukkan karakter tiap segmen.

In [4]:
profil = (df.groupby('klaster')
            .agg(jumlah=('harga', 'size'),
                 luas_tanah=('luas_m2', 'mean'),
                 luas_bangunan=('luas_bangunan', 'mean'),
                 kamar=('jumlah_kamar', 'mean'),
                 kamar_mandi=('jumlah_kamar_mandi', 'mean'),
                 median_harga=('harga', 'median'))
            .round(1))
profil['median_harga'] = profil['median_harga'].map(lambda v: f'Rp {v:,.0f}')
profil

,jumlah,luas_tanah,luas_bangunan,kamar,kamar_mandi,median_harga
klaster,,,,,,
0,12878,209.2,184.2,4.0,3.2,"Rp 2,000,000,000"
1,40761,103.1,63.0,2.1,1.3,"Rp 531,000,000"
2,2531,880.7,578.3,6.0,4.6,"Rp 5,950,000,000"


In [5]:
tabel = pd.crosstab(df['klaster'], df['kelas_harga'], normalize='index').round(3) * 100
print('Komposisi kelas harga per klaster (%):')
print(tabel.to_string())

ari = adjusted_rand_score(df['kelas_harga'], df['klaster'])
nmi = normalized_mutual_info_score(df['kelas_harga'], df['klaster'])
print(f'\nAdjusted Rand Index      : {ari:.4f}')
print(f'Normalized Mutual Info   : {nmi:.4f}')

Komposisi kelas harga per klaster (%):
kelas_harga  Mahal  Murah  Sedang
klaster                          
0             76.3    2.4    21.3
1             16.0   45.1    38.9
2             93.2    1.1     5.6

Adjusted Rand Index      : 0.1567
Normalized Mutual Info   : 0.2243


## Langkah 5 — Visualisasi Klaster dengan PCA

Empat dimensi tidak bisa digambar langsung, jadi data diproyeksikan ke dua komponen
utama (PCA). Sumbu ini hanya untuk melihat sebaran; klaster tetap dihitung pada empat
fitur asli.

In [6]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Z = pca.fit_transform(X)
df['pc1'], df['pc2'] = Z[:, 0], Z[:, 1]

print(f'Varians yang dijelaskan PC1+PC2: {pca.explained_variance_ratio_.sum():.2%}')

viz = df.sample(6000, random_state=RANDOM_STATE)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=viz, x='pc1', y='pc2', hue='klaster', palette='viridis',
                s=12, alpha=0.6, ax=ax[0], legend='full')
ax[0].set_title('Klaster K-Means (PCA 2D)')

sns.scatterplot(data=viz, x='pc1', y='pc2', hue='kelas_harga',
                hue_order=['Murah', 'Sedang', 'Mahal'],
                palette={'Murah': 'green', 'Sedang': 'orange', 'Mahal': 'red'},
                s=12, alpha=0.6, ax=ax[1])
ax[1].set_title('Kelas harga sebenarnya (PCA 2D)')

plt.tight_layout()
plt.show()

Varians yang dijelaskan PC1+PC2: 86.77%


## Langkah 6 — Tugas: Segmentasi dengan `k = 5`

Pada jobsheet asli, tugasnya adalah menerapkan K-Means pada `Mall_Customers.csv` dan
menentukan jumlah klaster. Di sini tugasnya: bagi properti menjadi **5 segmen** dan
jelaskan karakter masing-masing.

`k = 5` dipakai untuk melihat apakah segmen yang lebih halus masih bermakna secara bisnis
(mis. rumah kecil murah, rumah sedang, rumah besar, rumah sangat besar, properti mewah).

In [7]:
km5 = KMeans(n_clusters=5, n_init=10, random_state=RANDOM_STATE).fit(X)
df['klaster5'] = km5.labels_

profil5 = (df.groupby('klaster5')
             .agg(jumlah=('harga', 'size'),
                  luas_tanah=('luas_m2', 'mean'),
                  luas_bangunan=('luas_bangunan', 'mean'),
                  kamar=('jumlah_kamar', 'mean'),
                  median_harga=('harga', 'median'))
             .round(1)
             .sort_values('median_harga'))
profil5['median_harga'] = profil5['median_harga'].map(lambda v: f'Rp {v:,.0f}')
print(f'Silhouette k=5: {silhouette_score(X, df["klaster5"], sample_size=5000, random_state=RANDOM_STATE):.4f}')
profil5

Silhouette k=5: 0.4442


,jumlah,luas_tanah,luas_bangunan,kamar,median_harga
klaster5,,,,,
4,32420,98.4,55.7,1.9,"Rp 450,000,000"
1,16284,135.4,112.8,3.2,"Rp 1,300,000,000"
0,5553,328.4,289.7,4.8,"Rp 3,238,498,613"
3,359,501.9,478.7,12.8,"Rp 3,700,000,000"
2,1554,1109.2,632.1,4.4,"Rp 6,400,000,000"


In [8]:
plt.figure(figsize=(10, 5))
urut = df.groupby('klaster5')['harga'].median().sort_values().index
sns.boxplot(data=df, x='klaster5', y='harga', order=urut, showfliers=False)
plt.yscale('log')
plt.xlabel('Klaster (diurutkan dari median harga terendah)')
plt.ylabel('Harga (Rp, skala log)')
plt.title('Sebaran harga per segmen (k = 5)')
plt.tight_layout()
plt.show()

**Analisis JS04:**

- Silhouette **tertinggi pada k = 2 (0,643)** dan turun saat k naik (k = 3 → 0,514; k = 5 → 0,437). Kurva elbow juga tidak punya siku tajam: data properti tersebar kontinu, batas antarsegmen tidak setegas data blob/digit.
- `k = 3` tetap dipilih agar sebanding dengan tiga kelas harga. Hasilnya: silhouette 0,5247, Davies-Bouldin 1,048, tiga klaster berukuran **40.761 / 12.878 / 2.531** dengan median harga **Rp531 juta / Rp2,0 miliar / Rp5,95 miliar**.
- Klaster memang menangkap **ukuran properti**: klaster terkecil (2.531 rumah) punya rata-rata luas tanah 881 m² dan harga Rp5,95 M; klaster terbesar (40.761) luas 103 m² dan harga Rp531 juta.
- Namun kesesuaian dengan kelas harga **terbatas**: ARI 0,1567 dan NMI 0,2243. Komposisi menunjukkan klaster besar masih campur (45,1% Murah, 38,9% Sedang), sedangkan klaster mewah 93,2% Mahal tetapi hanya 2.531 rumah.
- Penyebab: harga tidak hanya ditentukan luas dan jumlah kamar, tetapi juga **lokasi**. Dua rumah berukuran sama bisa berbeda harga 5–10 kali antar kota. Karena `lokasi` tidak masuk fitur klaster, klaster tidak bisa menangkap perbedaan harga antarwilayah.
- Segmentasi `k = 5` (silhouette 0,444) memberi pembagian lebih rinci: 32.420 rumah murah (median Rp450 juta), 16.284 menengah (Rp1,3 M), 5.553 besar (Rp3,24 M), 359 rumah sangat besar (12,8 kamar, Rp3,7 M), dan 1.554 properti mewah berluas tanah >1.100 m² (Rp6,4 M).
- Kesimpulan: K-Means cocok untuk **segmentasi ukuran/profil properti**, bukan pengganti label harga. Prediksi harga tetap memerlukan model supervised (JS07/JS13).
